# Source-Weighted Evidence Mechanism Probe

- **Project:** Mass-communication opinion-leader model
- **Submodel ID and version:** `SM-OL-WEIGHT-01`
- **Framework link:** Reviewed `opleader` rough design
- **Probe type:** Mechanism probe
- **Date:** 2026-09-04
- **Status:** Runnable

## 1. Question, Decision, and Framework Link

- **Primary question:** What are the consequences of treating an opinion-leader message as more Beta evidence than a press message?
- **Decision supported:** Retain or revise source-weighted Beta evidence before adding production, delivery, and network reach.
- **Shared interfaces:** `Message`, `Exposure`, `MessageAggregation`, `MessageEvidence`, `OpinionUpdate`, and `BetaBelief`.
- **Highest intended claim level:** V1 isolated-mechanism behavior under fixed synthetic inputs.


## 2. Provenance and Boundary Contract

- Katz's research motivates source-mediated personal influence.
- Press production, leader production, delivery, network topology, topic knowledge, and all feedbacks are omitted.
- Each case performs one aggregation and one synchronous opinion update. There is no stochasticity or seed.
- Source weights are dimensionless multipliers of the common base evidence weight.


## 3. Expected Outcomes Before Running

- Equal source weights reproduce homogeneous baseline aggregation.
- A larger leader weight moves the posterior farther toward the leader stance.
- Larger weights also increase concentration, reducing later susceptibility.
- A more concentrated prior moves less in its mean under identical evidence.
- Passing these checks supports implementation consistency only; it does not validate leader credibility or the chosen weights.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / 'src' / 'opinion_model').is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from opinion_model.baseline import propose_opinion_update
from opinion_model.core import (
    AgentState, AggregationContext, BetaBelief, Exposure, Message,
)
from opinion_model.opleader import OriginatorKind, SourceWeightedAggregation

RUN = {
    'submodel_id': 'SM-OL-WEIGHT-01',
    'submodel_version': '0.1',
    'boundary_scenario': 'fixed one-round exposures',
    'base_evidence_weight': 1.0,
    'press_weight': 1.0,
    'leader_weights': [1.0, 2.0, 4.0],
    'active_feedbacks': [],
    'omitted_feedbacks': ['production', 'delivery', 'network', 'multi-round relay'],
}
RUN


## 4. Minimal Specification and Implementation

The notebook imports the tested scientific mechanism from `opinion_model.opleader`. The helper below only constructs fixed boundary inputs and collects outputs. Press uses an external originator ID; the leader uses an agent ID.


In [ ]:
PRESS_ID = 10
LEADER_ID = 1
CONSUMER_ID = 0

def make_exposure(producer_id, stance, label):
    return Exposure(
        round_index=1,
        consumer_id=CONSUMER_ID,
        message=Message(
            message_id=f'r1:{label}',
            round_index=1,
            producer_id=producer_id,
            stance=stance,
        ),
    )

def evaluate_case(prior, leader_weight, messages):
    aggregator = SourceWeightedAggregation(
        originator_kind_by_id={
            PRESS_ID: OriginatorKind.PRESS,
            LEADER_ID: OriginatorKind.LEADER,
        },
        source_weight_by_kind={
            OriginatorKind.PRESS: RUN['press_weight'],
            OriginatorKind.LEADER: leader_weight,
        },
    )
    exposures = tuple(
        make_exposure(producer_id, stance, label)
        for producer_id, stance, label in messages
    )
    evidence = aggregator(
        exposures, AggregationContext(RUN['base_evidence_weight'])
    )
    before = AgentState(BetaBelief(*prior))
    after = propose_opinion_update(before, evidence)
    return {
        'weighted_support': evidence.weighted_support,
        'weighted_oppose': evidence.weighted_oppose,
        'mean_before': before.belief.mean,
        'mean_after': after.belief.mean,
        'mean_change': after.belief.mean - before.belief.mean,
        'concentration_before': before.belief.concentration,
        'concentration_after': after.belief.concentration,
    }


## 5. Deterministic and Boundary Checks

These cases verify no-input invariance, source-weight direction, conflict resolution, and the associated concentration increase.


In [ ]:
empty = evaluate_case((2.0, 2.0), 3.0, [])
press_support = evaluate_case(
    (2.0, 2.0), 3.0, [(PRESS_ID, 1, 'press-support')]
)
leader_support = evaluate_case(
    (2.0, 2.0), 3.0, [(LEADER_ID, 1, 'leader-support')]
)
conflict = evaluate_case(
    (2.0, 2.0),
    3.0,
    [(PRESS_ID, 1, 'press-support'), (LEADER_ID, -1, 'leader-oppose')],
)

assert empty['mean_after'] == empty['mean_before']
assert empty['concentration_after'] == empty['concentration_before']
assert leader_support['mean_change'] > press_support['mean_change'] > 0.0
assert conflict['mean_after'] < conflict['mean_before']
assert conflict['concentration_after'] == 8.0

checks = pd.DataFrame(
    [empty, press_support, leader_support, conflict],
    index=['no exposure', 'press +', 'leader +', 'press + / leader -'],
)
checks.round(4)


## 6. Exploratory Experiment

The sweep varies leader source weight and prior concentration. It compares reinforcing messages, conflicting messages, and repeated leader messages. No parameter is calibrated.


In [ ]:
priors = {
    'uncertain neutral': (2.0, 2.0),
    'confident neutral': (20.0, 20.0),
    'positive leaning': (8.0, 2.0),
}
scenarios = {
    'reinforcement: press + / leader +': [
        (PRESS_ID, 1, 'press-support'),
        (LEADER_ID, 1, 'leader-support'),
    ],
    'conflict: press + / leader -': [
        (PRESS_ID, 1, 'press-support'),
        (LEADER_ID, -1, 'leader-oppose'),
    ],
    'repeat: two leader +': [
        (LEADER_ID, 1, 'leader-support-1'),
        (LEADER_ID, 1, 'leader-support-2'),
    ],
}

rows = []
for prior_name, prior in priors.items():
    for scenario_name, messages in scenarios.items():
        for leader_weight in RUN['leader_weights']:
            row = evaluate_case(prior, leader_weight, messages)
            rows.append({
                'prior': prior_name,
                'scenario': scenario_name,
                'leader_weight': leader_weight,
                **row,
            })

results = pd.DataFrame(rows)
conflict_results = results.query("scenario == 'conflict: press + / leader -'")
for _, group in conflict_results.groupby('prior'):
    ordered = group.sort_values('leader_weight')
    assert ordered['mean_after'].is_monotonic_decreasing
    assert ordered['concentration_after'].is_monotonic_increasing
results.round(4)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for prior_name, group in conflict_results.groupby('prior'):
    ordered = group.sort_values('leader_weight')
    axes[0].plot(
        ordered['leader_weight'], ordered['mean_after'], marker='o', label=prior_name
    )
    axes[1].plot(
        ordered['leader_weight'],
        ordered['concentration_after'],
        marker='o',
        label=prior_name,
    )
axes[0].axhline(0.5, color='black', linewidth=0.8, linestyle='--')
axes[0].set(title='Conflicting messages', xlabel='Leader source weight', ylabel='Posterior mean')
axes[1].set(title='Confidence accumulation', xlabel='Leader source weight', ylabel='Posterior concentration')
axes[0].legend(frameon=False)
fig.tight_layout()
plt.show()


## 7. Results and Conditional Interpretation

The next cell reports the observed deterministic result. Interpretation remains conditional on the fixed-message boundary and does not establish empirical validity.


In [ ]:
uncertain_conflict = conflict_results.query("prior == 'uncertain neutral'").sort_values('leader_weight')
confident_conflict = conflict_results.query("prior == 'confident neutral'").sort_values('leader_weight')
low = uncertain_conflict.iloc[0]
high = uncertain_conflict.iloc[-1]
confident_high = confident_conflict.iloc[-1]
display(Markdown(
    f"""
- **Observed:** With equal press and leader weights, opposing messages leave the uncertain neutral posterior mean at `{low['mean_after']:.3f}`. Increasing leader weight to `{high['leader_weight']:.0f}` moves it to `{high['mean_after']:.3f}` and raises concentration from `{low['concentration_after']:.1f}` to `{high['concentration_after']:.1f}`.
- **Prior dependence:** Under the same highest-weight conflict, the confident neutral prior ends at `{confident_high['mean_after']:.3f}`, closer to its initial mean than the uncertain prior.
- **Highest completed level:** V1 behavior of source-weighted aggregation coupled to the existing Beta update under fixed inputs.
- **Supports:** The implementation realizes the selected weighted-evidence interpretation and exposes its joint movement-and-confidence consequence.
- **Does not support:** Any empirical leader weight, network effect, population outcome, or claim that leaders are truly more credible.
"""
))


## 8. Disposition and Change Impact

- **Disposition:** Retain as a benchmark pending researcher review.
- **Affected mechanism:** `opleader` message aggregation only.
- **Unaffected:** Baseline aggregation, shared contracts, scheduler, production, selection, network, and platform case.
- **Next decision if retained:** Whether the joint increase in opinion movement and confidence matches the intended meaning of leader influence.
- **Required later check:** V2 coupling only after press production, leader production, and delivery interfaces are specified.
